# Bagging & Random Forests

**Companion lesson:** https://ml-viz.vercel.app/courses/ensemble-methods/01-bagging-and-random-forests

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Bootstrap Aggregating

Each tree trains on a random sample with replacement. Averaging reduces variance.

In [ ]:
np.random.seed(42)
n = 100
X = np.sort(5 * np.random.rand(n))
y_true = np.sin(X) + 0.3 * np.random.randn(n)

def fit_stump(X, y):
    best_loss, best_t, best_v = np.inf, 0, 0
    for t in np.unique(X):
        for v in [y[X <= t].mean(), y[X > t].mean()]:
            pred = np.where(X <= t, y[X <= t].mean(), y[X > t].mean())
            loss = np.mean((y - pred)**2)
            if loss < best_loss:
                best_loss, best_t = loss, t
    return best_t

def predict_stump(x, X, y, t):
    return np.where(x <= t, y[X <= t].mean(), y[X > t].mean())

# Bootstrap samples
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x_grid = np.linspace(0, 5, 200)

for ax_i in range(3):
    ax = axes[ax_i]
    idx = np.random.choice(n, n, replace=True)
    t = fit_stump(X[idx], y_true[idx])
    pred = predict_stump(x_grid, X[idx], y_true[idx], t)
    ax.scatter(X, y_true, c='#818cf8', s=10, alpha=0.4)
    ax.plot(x_grid, pred, color='#f43f5e', linewidth=2)
    ax.set_title(f'Bootstrap Tree {ax_i+1}', color='white', fontsize=11)

plt.suptitle('Each Tree Sees Different Data', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Average of many trees
fig, ax = plt.subplots(figsize=(10, 5))
predictions = []
for _ in range(50):
    idx = np.random.choice(n, n, replace=True)
    t = fit_stump(X[idx], y_true[idx])
    pred = predict_stump(x_grid, X[idx], y_true[idx], t)
    predictions.append(pred)
    ax.plot(x_grid, pred, color='#818cf8', alpha=0.08, linewidth=0.5)

ensemble = np.mean(predictions, axis=0)
ax.plot(x_grid, np.sin(x_grid), color='#94a3b8', linestyle='--', linewidth=1.5, label='True function')
ax.plot(x_grid, ensemble, color='#14b8a6', linewidth=2.5, label='Ensemble (50 trees)')
ax.scatter(X, y_true, c='#818cf8', s=10, alpha=0.3)
ax.legend()
ax.set_title('Bagging: Averaging Reduces Variance', color='white')
plt.tight_layout()
plt.show()

## The two numbers behind bagging

**(1) Out-of-bag fraction.** A bootstrap draw misses a given row with probability $(1-1/n)^n \to e^{-1}\approx 0.368$ — so ~37% of rows are OOB for each tree.

**(2) Variance of an average of correlated trees.** For $B$ trees with variance $\sigma^2$ and pairwise correlation $\rho$, the ensemble variance is $\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$. More trees only kill the second term; the floor $\rho\sigma^2$ falls only by **decorrelating** (what random forests do). We verify both by simulation.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# (1) OOB fraction: closed form vs empirical
print('Out-of-bag fraction:')
for n in [5, 20, 100, 1000]:
    closed = (1 - 1/n)**n
    # empirical: fraction of rows never drawn across many bootstraps
    miss = np.mean([len(set(range(n)) - set(rng.integers(0, n, n))) / n for _ in range(2000)])
    print(f'  n={n:4d}: (1-1/n)^n = {closed:.4f}   empirical OOB = {miss:.4f}')
print(f'  limit 1/e = {np.exp(-1):.4f}\n')

# (2) Variance of average of B correlated "trees": rho*sigma^2 + (1-rho)/B * sigma^2
# Build correlated unit-variance vars: T_b = sqrt(rho)*Z_common + sqrt(1-rho)*Z_b
def ensemble_var(rho, B, trials=20000):
    Zc = rng.normal(size=(trials, 1))
    Zb = rng.normal(size=(trials, B))
    T = np.sqrt(rho) * Zc + np.sqrt(1 - rho) * Zb     # var(T_b)=1, corr=rho
    return T.mean(axis=1).var()

print('Ensemble variance (sigma^2 = 1):  formula = rho + (1-rho)/B')
for rho in [0.5, 0.1]:
    print(f'  rho={rho}:')
    for B in [1, 10, 1000]:
        formula = rho + (1 - rho) / B
        print(f'    B={B:4d}: formula={formula:.3f}  simulated={ensemble_var(rho, B):.3f}')
print('\n-> more trees (B up) only removes (1-rho)/B; the floor = rho is lowered only by decorrelating.')


## Out-of-bag evaluation and feature importance

Each bootstrap sample leaves out ~37% of rows — the **out-of-bag** set gives a free validation estimate. Random forests also rank features by impurity reduction.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0)
rf.fit(data.data, data.target)
print('OOB accuracy:', round(rf.oob_score_, 4))

imp = sorted(zip(rf.feature_importances_, data.feature_names), reverse=True)[:5]
print('top features:')
for score, name in imp:
    print(f'  {name:25s} {score:.3f}')

## Key takeaways

- **Bagging** trains models on bootstrap samples and averages them — cutting **variance**.
- **Random forests** add per-split feature randomness to decorrelate the trees.
- **Out-of-bag** rows give a built-in validation score (no separate hold-out needed).
- Feature importances rank inputs, but prefer **permutation importance** for reliability.